# 🏡 California House Price Prediction

End-to-end ML project using the California Housing dataset.  
Models: Linear Regression, Decision Tree, Random Forest + GridSearchCV.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from sklearn.model_selection import train_test_split, StratifiedShuffleSplit, cross_val_score, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

## 1. Data Loading & Exploration

In [ ]:
df = pd.read_csv('housing.csv')
print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Distribution of categorical feature
df['ocean_proximity'].value_counts()

In [ ]:
# Histograms of all numeric features
df.hist(bins=50, figsize=(20, 15))
plt.suptitle("Feature Distributions", fontsize=16)
plt.tight_layout()
plt.show()

## 2. Stratified Train/Test Split

Using `median_income` categories to ensure a representative split.

In [ ]:
df["income_cat"] = pd.cut(df["median_income"],
                          bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
                          labels=[1, 2, 3, 4, 5])
df["income_cat"].hist()
plt.title("Income Category Distribution")
plt.show()

In [ ]:
split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_index, test_index in split.split(df, df["income_cat"]):
    strat_train_set = df.loc[train_index]
    strat_test_set  = df.loc[test_index]

for set_ in (strat_train_set, strat_test_set):
    set_.drop("income_cat", axis=1, inplace=True)

print(f"Train size: {len(strat_train_set)} | Test size: {len(strat_test_set)}")

## 3. Exploratory Data Analysis (EDA)

In [ ]:
housing = strat_train_set.copy()

housing.plot(kind="scatter", x="longitude", y="latitude", alpha=0.4,
             s=housing["population"] / 100, label="population", figsize=(10, 7),
             c=housing["median_house_value"], cmap=plt.get_cmap("jet"), colorbar=True)
plt.legend()
plt.title("Housing Prices by Location")
plt.show()

In [ ]:
corr_matrix = housing.corr(numeric_only=True)
corr_matrix["median_house_value"].sort_values(ascending=False)

In [ ]:
from pandas.plotting import scatter_matrix
attributes = ["median_house_value", "median_income", "total_rooms", "housing_median_age"]
scatter_matrix(housing[attributes], figsize=(12, 8))
plt.suptitle("Scatter Matrix — Top Correlated Features", fontsize=14)
plt.show()

## 4. Feature Engineering

In [ ]:
housing["rooms_per_household"]     = housing["total_rooms"]    / housing["households"]
housing["bedrooms_per_room"]        = housing["total_bedrooms"] / housing["total_rooms"]
housing["population_per_household"] = housing["population"]     / housing["households"]

corr_matrix = housing.corr(numeric_only=True)
corr_matrix["median_house_value"].sort_values(ascending=False)

## 5. Preprocessing Pipeline

In [ ]:
df = strat_train_set.drop("median_house_value", axis=1)
df_labels = strat_train_set["median_house_value"].copy()

num_attribs = list(df.drop("ocean_proximity", axis=1))
cat_attribs = ["ocean_proximity"]

# Numeric: impute missing values, then scale
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])

# Full pipeline: numeric + one-hot encoding for categorical
my_new_pipeline = ColumnTransformer([
    ("num", num_pipeline,    num_attribs),
    ("cat", OneHotEncoder(), cat_attribs),
])

df_complete_tr = my_new_pipeline.fit_transform(df)
print(f"Transformed training set shape: {df_complete_tr.shape}")

## 6. Model Training & Cross-Validation

Comparing Linear Regression, Decision Tree, and Random Forest using 10-fold CV.

In [ ]:
def display_scores(scores, model_name=""):
    print(f"Model: {model_name}")
    print(f"  RMSE scores: {scores.round(0)}")
    print(f"  Mean:        {scores.mean():.0f}")
    print(f"  Std:         {scores.std():.0f}")
    print()

In [ ]:
lg_model = LinearRegression()
lg_model.fit(df_complete_tr, df_labels)

lin_scores = cross_val_score(lg_model, df_complete_tr, df_labels,
                             scoring="neg_mean_squared_error", cv=10)
display_scores(np.sqrt(-lin_scores), "Linear Regression")

In [ ]:
df_tree_model = DecisionTreeRegressor()
df_tree_model.fit(df_complete_tr, df_labels)

tree_scores = cross_val_score(df_tree_model, df_complete_tr, df_labels,
                              scoring="neg_mean_squared_error", cv=10)
display_scores(np.sqrt(-tree_scores), "Decision Tree")

In [ ]:
rf_model = RandomForestRegressor(random_state=42)
rf_model.fit(df_complete_tr, df_labels)

for_scores = cross_val_score(rf_model, df_complete_tr, df_labels,
                             scoring="neg_mean_squared_error", cv=10)
display_scores(np.sqrt(-for_scores), "Random Forest")

## 7. Hyperparameter Tuning — GridSearchCV

In [ ]:
param_grid = [
    {"n_estimators": [3, 10, 30], "max_features": [2, 4, 6, 8]},
    {"bootstrap": [False], "n_estimators": [3, 10], "max_features": [2, 3, 4]},
]

forest_reg = RandomForestRegressor(random_state=42)
grid_search = GridSearchCV(forest_reg, param_grid, cv=5,
                           scoring="neg_mean_squared_error")
grid_search.fit(df_complete_tr, df_labels)

print("Best parameters:", grid_search.best_params_)

In [ ]:
cvres = grid_search.cv_results_
print(f"{'RMSE':<12} Params")
print("-" * 60)
for mean_score, params in zip(cvres["mean_test_score"], cvres["params"]):
    print(f"{np.sqrt(-mean_score):<12.0f} {params}")

## 8. Final Evaluation on Test Set

In [ ]:
final_model = grid_search.best_estimator_

X_test          = strat_test_set.drop("median_house_value", axis=1)
y_test          = strat_test_set["median_house_value"].copy()
X_test_prepared = my_new_pipeline.transform(X_test)

final_predictions = final_model.predict(X_test_prepared)
final_mse         = mean_squared_error(y_test, final_predictions)
final_rmse        = np.sqrt(final_mse)

# 95% Confidence Interval
squared_errors = (final_predictions - y_test) ** 2
interval = np.sqrt(stats.t.interval(
    confidence=0.95,
    df=len(squared_errors) - 1,
    loc=squared_errors.mean(),
    scale=stats.sem(squared_errors),
))

print("=" * 42)
print(f"  Final RMSE:              {final_rmse:,.0f}")
print(f"  95% Confidence Interval: {interval[0]:,.0f} – {interval[1]:,.0f}")
print("=" * 42)